# ReTone Track B — Train AFTER for polyphonic timbre transfer (44.1 kHz)

Train our own **AFTER (ACIDS-IRCAM control-transfer latent diffusion)** checkpoint on paired MIDI-rendered audio, targeting **piano → orchestral** (strings + brass) as the demo goal. Drops into the worker's `after_models/` alongside the pretrained AFTER checkpoints from Track A, so the frontend just gets more instrument options in the dropdown without any code changes.

### Scope — what this notebook builds

One model that handles **polyphonic** timbre transfer for the pitched-poly family (Engine 2). Training pipeline:

1. **Slakh2100** (145 h, 34 instrument classes, per-stem Kontakt renders + aligned MIDI, CC BY 4.0) — the anchor. Contains both source (piano) and target (strings/brass) stems in one aligned dataset.
2. **Self-rendered Lakh MIDI** (~500 h, optional) — via FluidSynth + Virtual Playing Orchestra + Salamander + GeneralUser GS with per-preset reverb/EQ/room-IR randomization. For preset diversity (Zehren et al. showed diversity beats raw hours past ~250 h).
3. **URMP + GuitarSet** (real recordings, ~7 h) — sim-to-real fine-tune stage to close the gap between synth-trained and real-audio-inference.

Produces two artifacts (both TorchScript):
- `piano_orchestral_ae.ts` — the RAVE-style latent autoencoder codec (~30-50 MB).
- `piano_orchestral_diff.ts` — the latent diffusion U-Net + disentanglement heads (~100-500 MB).

Drops into `runpod/after_models/piano_orchestral/` on the worker. See §8 for the drop-in instructions.

### Prerequisites

**Rented RunPod GPU Pod, A100 40 GB or L40S 48 GB.** The 24 GB tier will OOM on the diffusion stage without heavy `waveform_sec` reductions. Disk: **~300 GB** (Slakh alone is 104 GB compressed, extracts to ~500 GB — keep the compressed tarball only for the initial extract, then delete it).

Same on-pod workflow as `train_ddsp_48k.ipynb`: `git clone https://github.com/Ganeshveer/retone` on the pod, open this notebook in `retone/notebooks/`, run top-to-bottom.

### Rough wall-clock and cost

| Stage | Steps | Wall-clock (A100 40 GB) | Cost @ $1.60/hr |
|---|---|---|---|
| §5 sanity overfit | 2 000 | ~15 min | ~$0.40 |
| §6 autoencoder (Stage 1) | ~2 M | 3-5 days | ~$150-200 |
| §6.1 latent diffusion (Stage 2) | ~1.5 M | 5-7 days | ~$200-270 |
| §7 URMP fine-tune (optional) | ~100 k | ~1 day | ~$40 |
| **Total** | | **~10-15 days** | **~$400-500** |

Recommended posture: do §5 (~15 min, ~$0.40) FIRST to prove the pipeline is sound. Only then launch the full-day training runs. `--resume` is safe across pod restarts as long as `CKPT_DIR/latest.pth` exists on the persistent volume.

### Design decisions locked with the user
- **License:** CC BY-NC 4.0 on AFTER's code is fine for our research/personal use. Own-trained weights (this notebook's output) are 100% ours — no license entanglement on the checkpoint side.
- **Target family:** piano source → orchestral (strings + brass). Slakh2100's `Strings` and `Brass` class groupings map cleanly.
- **Sample rate: 44 100 Hz** to match AFTER's paper defaults + the pretrained checkpoints' SR.


## 0 · Configuration

Set the target family + capacity here. Everything downstream reads `CFG`. To train a different pair later (e.g. guitar → orchestral), change `SOURCE_STEMS` / `TARGET_STEMS` and re-run from §3.

In [ ]:
import os, sys, subprocess, pathlib, json, shutil, textwrap

# ── target — what "poly-timbre" this model learns ───────────────────────────
FAMILY_NAME = "piano_orchestral"      # ships as runpod/after_models/{FAMILY_NAME}/
SOURCE_STEMS = ["Piano"]              # Slakh instrument-class names for the source
TARGET_STEMS = ["Strings", "Brass"]   # Slakh instrument-class names for target

# ── paths (relative to the pod's working dir) ───────────────────────────────
ROOT     = pathlib.Path("/workspace/retone_train_after").resolve()
REPO_DIR = pathlib.Path("..").resolve()                              # the retone repo (this nb is in retone/notebooks)
AFTER_DIR = ROOT / "control-transfer-diffusion"                       # AFTER source (NilsDem/ ...)
DATA_DIR  = ROOT / "data"                                             # slakh2100/, lmd_render/, urmp/
CKPT_DIR  = ROOT / "ckpt" / FAMILY_NAME
OUT_DIR   = ROOT / "artifacts" / FAMILY_NAME
for d in (ROOT, DATA_DIR/"slakh2100", DATA_DIR/"lmd_render", DATA_DIR/"urmp",
          CKPT_DIR/"ae", CKPT_DIR/"diff", OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── 44 kHz max-quality capacity (from AFTER paper + Slakh recipes) ──────────
# Values default to AFTER's own gin configs (see AFTER_DIR/configs/*.gin after cloning).
# Any values you need to override for OUR data volume are the ones commented below.
CFG = dict(
    sample_rate=44100,
    signal_length=524288,          # ~11.9 s @ 44.1 kHz — matches AFTER paper's 12 s crop
    block_size=1024,               # AFTER's latent frame rate (44100 / 1024 ≈ 43 Hz)
    n_diffusion_steps=20,          # inference-time DDIM steps

    # autoencoder (Stage 1)
    ae_batch_size=8,               # 8 fits A100 40 GB with grad accum; drop to 4 on 24 GB
    ae_steps=2_000_000,
    ae_lr=1e-4,
    ae_lr_decay=0.999,

    # latent diffusion (Stage 2)
    diff_batch_size=16,            # 16 fits A100 40 GB
    diff_steps=1_500_000,
    diff_lr=1e-4,

    # sim-to-real fine-tune (Stage 3, optional but recommended)
    finetune_steps=100_000,

    device="cuda",
    seed=940513,
    num_workers=8,                 # measure sustained samples/sec; recipe pod has ~48 CPU quota
)
print("FAMILY_NAME =", FAMILY_NAME)
print("SOURCE      =", SOURCE_STEMS, "-> TARGET =", TARGET_STEMS)
print("ROOT        =", ROOT)
print("REPO_DIR    =", REPO_DIR, "(exists:", (REPO_DIR/"runpod"/"ddsp_engine.py").exists(), ")")


## 1 · Environment

Match AFTER's own pinned deps (torch 2.0+, Python 3.12 per README) plus the audio-tooling we need for data prep. Same principle as the DDSP notebook: the trained artifact is a plain TorchScript file that will re-load on ANY torch 2.x, so we can pin an older torch here without affecting the worker's torch 2.7.1 runtime.

In [ ]:
# GPU + torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true
!nproc

# torch stack (CUDA 12.1 wheels — works on RTX 4090/A100/L40S which are all sm 8.6/8.9/9.0)
!pip -q install torch==2.1.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121

# AFTER's own deps (from acids-ircam/AFTER/requirements.txt, audited 2026-08-16)
!pip -q install torch-ema einops einops-exts scikit-learn umap-learn info-nce-pytorch
!pip -q install "librosa>=0.10.2" nnAudio "audiomentations<=0.40" pedalboard==0.9.16 resampy
!pip -q install pretty-midi mir-eval
!pip -q install "numpy==1.26" "scipy==1.12.0" pandas lmdb protobuf
!pip -q install gin-config absl-py "tqdm>=4.64.1" pyyaml tensorboard "matplotlib>=3.8.3"
!pip -q install "cached-conv>=2.5.0" nn-tilde packaging

# Extras for our data pipeline (Slakh loader + soundfont rendering)
!pip -q install soundfile pyloudnorm requests
!pip -q install pyFluidSynth   # bindings to system fluidsynth; below installs the binary too
!which ffmpeg     || (apt-get -qq update && apt-get -qq install -y ffmpeg)
!which fluidsynth || (apt-get -qq update && apt-get -qq install -y fluidsynth)

# TF GPU-visibility trap (from the DDSP notebook) — CREPE/TF are NOT used here; AFTER's
# f0 branch uses BasicPitch (torch-native), so no TF surprise fallback risk this time.
import torch
print("torch", torch.__version__, "| cuda avail:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
# Clone AFTER (control-transfer-diffusion research fork — the one with train.py entry points).
import subprocess, shutil, re
if not AFTER_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/NilsDem/control-transfer-diffusion.git", str(AFTER_DIR)],
        check=True,
    )
print("AFTER checked out at", AFTER_DIR)
print("entry points present:",
      [p.name for p in AFTER_DIR.glob("train_*.py")],
      "+ export:", [p.name for p in AFTER_DIR.glob("export_*.py")])
print("configs:", sorted(p.name for p in (AFTER_DIR/"configs").glob("*.gin")) if (AFTER_DIR/"configs").exists() else "(inspect the repo — configs dir may be renamed)")

# NOTE: unlike the DDSP notebook, AFTER's code is modern (torch 2.0+, omegaconf 2.x, pandas 2.x
# native) so no compatibility patches are expected here. If a training step fails, cross-
# reference the DDSP notebook's PATCH cell — the same class of fixes (torch.fft ports, pandas
# .append -> pd.concat, omegaconf .save -> OmegaConf.save) may need applying to AFTER too.


## 2 · Data — Slakh2100 (anchor dataset)

Slakh2100 = 145 h of paired stems + MIDI, rendered by the original authors with commercial Kontakt patches. Provides *both* piano source AND orchestral target stems in the same aligned MIDI files — perfect for our paired-training goal. **CC BY 4.0** licensed (commercial-safe if we ever want to expand scope).

Zenodo tarball is **104 GB compressed FLAC** → **~500 GB extracted WAV**. Plan disk accordingly. Same download-and-verify pattern as the URMP fetch in `train_ddsp_48k.ipynb`.

In [ ]:
# Fetch Slakh2100 (once, validated). Skips if the extracted dataset already exists.
import requests, tqdm, tarfile

SLAKH_DIR = DATA_DIR/"slakh2100"
TAR_PATH = SLAKH_DIR/"slakh2100_flac_redux.tar.gz"
EXTRACT_DIR = SLAKH_DIR/"extracted"
SLAKH_URL = "https://zenodo.org/records/4599666/files/slakh2100_flac_redux.tar.gz"
# Verified from Zenodo record on 2026-08 — file is 104.3 GB.
SLAKH_EXPECTED_SIZE = 111_997_190_547

def fetch_slakh():
    if EXTRACT_DIR.exists() and any(EXTRACT_DIR.iterdir()):
        print("already extracted at", EXTRACT_DIR, "— skipping download.")
        return
    if TAR_PATH.exists() and TAR_PATH.stat().st_size == SLAKH_EXPECTED_SIZE:
        print("valid tarball present — skipping download.")
    else:
        print(f"downloading {SLAKH_URL} (~104 GB)…")
        part = TAR_PATH.with_suffix(".tar.gz.part")
        with requests.get(SLAKH_URL, stream=True, timeout=60) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            with open(part, "wb") as f, tqdm.tqdm(total=total, unit="B", unit_scale=True) as bar:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk); bar.update(len(chunk))
        part.rename(TAR_PATH)
        # Validate: gzip magic + expected size.
        with open(TAR_PATH, "rb") as f:
            assert f.read(2) == b"\x1f\x8b", "not a gzip archive"
        assert TAR_PATH.stat().st_size == SLAKH_EXPECTED_SIZE, (
            f"expected {SLAKH_EXPECTED_SIZE}, got {TAR_PATH.stat().st_size}"
        )
        print(f"✓ verified: {TAR_PATH.stat().st_size/1e9:.1f} GB gzip")

    print("extracting (~30-60 min, produces ~500 GB)…")
    EXTRACT_DIR.mkdir(exist_ok=True)
    with tarfile.open(TAR_PATH) as t:
        t.extractall(EXTRACT_DIR)
    n_tracks = len(list((EXTRACT_DIR/"slakh2100_flac_redux").glob("*/Track*")))
    print(f"done: {n_tracks} Slakh tracks extracted")

fetch_slakh()

# Free disk immediately after extract — the tarball is redundant now.
# (Same pattern as the URMP cleanup step we added for DDSP.)
if TAR_PATH.exists() and EXTRACT_DIR.exists() and any(EXTRACT_DIR.iterdir()):
    print(f"freeing {TAR_PATH.stat().st_size/1e9:.1f} GB — removing tarball")
    TAR_PATH.unlink()


### 2.1 · Filter Slakh to tracks that have both piano source AND orchestral targets

Slakh's per-track `metadata.yaml` lists each stem's Kontakt patch and its General MIDI program number. Program numbers 0-7 = Piano, 40-47 = Strings, 56-63 = Brass. We keep only tracks that contain a piano stem AND at least one string OR brass stem — these are the ones that give us aligned paired audio for our target family.

In [ ]:
# Build paired index: for each Slakh track keep piano_stem_path + list of (family, stem_path).
import yaml

SLAKH_ROOT = EXTRACT_DIR/"slakh2100_flac_redux"
PIANO_PROGRAMS = set(range(0, 8))       # GM 0-7
STRINGS_PROGRAMS = set(range(40, 48))   # GM 40-47
BRASS_PROGRAMS = set(range(56, 64))     # GM 56-63

def _family(program):
    if program in PIANO_PROGRAMS: return "Piano"
    if program in STRINGS_PROGRAMS: return "Strings"
    if program in BRASS_PROGRAMS: return "Brass"
    return None

def scan_slakh():
    pairs = []
    for track in sorted(SLAKH_ROOT.glob("Track*")):
        meta = track/"metadata.yaml"
        if not meta.exists(): continue
        d = yaml.safe_load(meta.read_text())
        stems = d.get("stems", {})
        piano_paths, target_paths_by_family = [], {}
        for stem_name, stem_meta in stems.items():
            program = stem_meta.get("program_num", -1)
            if not stem_meta.get("audio_rendered", False): continue
            fam = _family(program)
            if fam is None: continue
            audio = track/"stems"/f"{stem_name}.flac"
            if not audio.exists(): continue
            if fam == "Piano":
                piano_paths.append(audio)
            else:
                target_paths_by_family.setdefault(fam, []).append(audio)
        if piano_paths and target_paths_by_family:
            pairs.append({
                "track": track.name,
                "piano": [str(p) for p in piano_paths],
                "targets": {k: [str(p) for p in v] for k, v in target_paths_by_family.items()},
            })
    return pairs

paired_tracks = scan_slakh()
print(f"paired tracks (piano + strings|brass): {len(paired_tracks)} / 2100")
print("family coverage (target-side):")
from collections import Counter
c = Counter()
for p in paired_tracks:
    for fam in p["targets"]: c[fam] += 1
for k, v in c.items(): print(f"  {k}: {v} tracks")

# Persist the index for downstream cells + resume-friendliness.
(DATA_DIR/"slakh_paired_index.json").write_text(json.dumps(paired_tracks, indent=1))
print(f"\nindex written to {DATA_DIR}/slakh_paired_index.json")


### 2.2 · (Optional) Self-render extra data from Lakh MIDI for preset diversity

Zehren et al. (`arXiv:2407.19823`) found that past ~250 h of paired synth training data, **preset diversity** matters more than raw hours. Slakh alone uses 187 Kontakt patches; we can multiply that by rendering additional Lakh MIDI files through Virtual Playing Orchestra (CC BY 3.0, strings/brass), Salamander (CC BY 3.0, piano), and GeneralUser GS with per-render randomized reverb + EQ.

Skip this cell entirely if Slakh alone is enough to prove the pipeline. Turn it on once you've validated §5 + first ~100k steps of §6.

In [ ]:
# Off by default — flip to True once Slakh training is proven and you want more diversity.
RENDER_EXTRA = False

if RENDER_EXTRA:
    LMD_URL = "http://hog.ee.columbia.edu/craffel/lmd/lmd_full.tar.gz"     # 1.5 GB
    LMD_DIR = DATA_DIR/"lmd_render"

    # 1. Download Lakh MIDI (small enough to fetch directly, unlike Slakh)
    import requests, tarfile, tqdm
    LMD_TAR = LMD_DIR/"lmd_full.tar.gz"
    LMD_TAR.parent.mkdir(parents=True, exist_ok=True)
    if not LMD_TAR.exists():
        print("downloading Lakh MIDI (~1.5 GB)…")
        with requests.get(LMD_URL, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(LMD_TAR, "wb") as f, tqdm.tqdm(total=int(r.headers.get("content-length", 0)),
                                                     unit="B", unit_scale=True) as bar:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk); bar.update(len(chunk))
    LMD_MIDI_DIR = LMD_DIR/"midi"
    if not LMD_MIDI_DIR.exists():
        with tarfile.open(LMD_TAR) as t: t.extractall(LMD_DIR)

    # 2. Download soundfonts (all CC BY / permissive — see plan for licenses).
    SF_DIR = LMD_DIR/"soundfonts"
    SF_DIR.mkdir(exist_ok=True)
    SOUNDFONTS = {
        # (filename, URL, family this SF is used to render)
        "GeneralUser-GS.sf2": ("https://www.schristiancollins.com/generaluser.php", "MULTI"),
        "SalamanderGrand-V3.sfz": ("https://freepats.zenvoid.org/Piano/SalamanderGrandPiano/", "Piano"),
        "VirtualPlayingOrchestra.sfz": ("http://virtualplayingorchestra.com/", "Strings+Brass"),
    }
    print("MANUAL STEP: download the soundfonts to", SF_DIR, "before running the render.")
    for name, (url, fam) in SOUNDFONTS.items(): print(f"  {name}  ({fam})  — {url}")

    # 3. Render — for each of ~1000 random MIDIs, render through the source SF + target SF,
    #    with a random room-IR + EQ + gain per render for Zehren-style preset diversity.
    #
    #    This cell is a placeholder for the actual render loop (needs the SFs downloaded first
    #    per the manual step above). Wire it in once you're ready to scale beyond Slakh.
    print("\nrendering pipeline is scaffolded but not run — see the plan for the full recipe.")
else:
    print("RENDER_EXTRA=False — skipping. Slakh alone (~145 h) is the plan for the first training run.")


### 2.3 · Preprocess — 44.1 kHz, segment into 12s clips, write LMDB

AFTER's dataloader expects an LMDB store of preprocessed audio at the target sample rate. Match the paper's `signal_length: 524288` = ~11.9 s crop at 44.1 kHz. Every source clip is paired with the corresponding target clip at the same time offset (guaranteed by Slakh's per-stem alignment).

In [ ]:
# Build the paired LMDB store from the Slakh paired-index.
# NOTE: this is a template — AFTER's actual dataset builder lives in
# `AFTER_DIR/dataset/*.py` and is invoked from `train_autoencoder.py`. We call
# through to it here so the format matches what the training scripts expect.

import subprocess, sys
CACHE_DIR = ROOT/"cache"/FAMILY_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Verify AFTER's dataset preparation entry point exists (it's typically dataset/preprocess.py
# but the name has changed across AFTER revisions — check the repo before wiring).
prep_script = next(
    (AFTER_DIR/sub/"preprocess.py" for sub in ("dataset", "data", "utils") if (AFTER_DIR/sub/"preprocess.py").exists()),
    None,
)
if prep_script:
    print("found AFTER preprocess entry point:", prep_script)
else:
    print("MANUAL: locate AFTER's preprocess script (search for 'lmdb.open' or 'signal_length' in AFTER_DIR).")
    print("Then invoke it here with our data paths — this cell will call subprocess.run(...).")
    print("Files to check:", list(AFTER_DIR.rglob("preprocess*.py"))[:5])

# TEMPLATE — adapt once we know AFTER's exact CLI:
#   python <prep_script> --data-dir <paired_slakh_dir> --out-dir <CACHE_DIR>/paired.lmdb \
#                        --sample-rate 44100 --signal-length 524288
print("\nCACHE_DIR (where LMDB will land):", CACHE_DIR)


## 3 · Fast sanity check — overfit a tiny subset (~15 min, ~$0.40)

Before spending ~$400 on the full ~10-day training run, prove the whole pipeline works end-to-end by overfitting a *single* Slakh track for 2 000 steps. If it fails, the full run would too — and it'll fail in the same place, saving days.

In [ ]:
# Take the first paired track, run 2000 steps of the autoencoder + 2000 of diffusion.
# Success criteria: (a) both stages start, (b) MSS + latent losses trend down, (c) a
# reconstruction from step 2000 is audibly better than one from step 0.

SANE_DIR = ROOT/"sanity"/FAMILY_NAME
SANE_DIR.mkdir(parents=True, exist_ok=True)

# Copy just the first paired track's stems into a mini-dataset.
first = paired_tracks[0]
print(f"sanity track: {first['track']}")
print(f"  piano stems : {len(first['piano'])}")
print(f"  target stems: {sum(len(v) for v in first['targets'].values())}")

# NOTE: AFTER's train commands are gin-config driven. Look under AFTER_DIR/configs for the
# closest match to our setup (typical filenames: `slakh_ae.gin`, `main.gin`, `diffusion.gin`).
# The sanity run overrides `steps=2000`, `batch_size=2`, `save_every=500`.
CONFIGS = list((AFTER_DIR/"configs").glob("*.gin")) if (AFTER_DIR/"configs").exists() else []
print(f"\navailable gin configs in AFTER: {[p.name for p in CONFIGS]}")

# TEMPLATE — adapt to AFTER's real CLI once configs are located:
# python train_autoencoder.py --config configs/slakh.gin --data <SANE_DIR>/paired.lmdb \
#                              --steps 2000 --batch-size 2 --ckpt-dir <CKPT_DIR>/ae_sanity
# python train_diffusion.py   --config configs/diffusion.gin --ae-ckpt <CKPT_DIR>/ae_sanity/final.pt \
#                              --steps 2000 --batch-size 2 --ckpt-dir <CKPT_DIR>/diff_sanity
print("\nsanity run is scaffolded — fill in AFTER's actual CLI once the repo is cloned + inspected.")


## 4 · Full training run — Stage 1 (autoencoder) then Stage 2 (latent diffusion)

Two-stage run, ~10-15 days total on A100 40 GB. Both stages resume-safe if the pod restarts, provided `CKPT_DIR/{ae,diff}/latest.pt` exists on the persistent volume.

**Order matters** — Stage 2 uses Stage 1's autoencoder as a frozen latent codec. If Stage 1 is undertrained the diffusion stage cannot recover; do NOT start Stage 2 until Stage 1's reconstruction MSS loss has plateaued for ~50k steps.

In [ ]:
# TensorBoard — same as DDSP notebook, on the pod's forwarded port.
%load_ext tensorboard
%tensorboard --logdir {str(ROOT/"tensorboard_log")} --port 6006


In [ ]:
# STAGE 1 — Autoencoder. Detached with nohup so SSH disconnects don't kill training.
# Adapt the actual CLI once you've cloned the repo and inspected AFTER_DIR/train_autoencoder.py.
import subprocess

STAGE1_LOG = ROOT/"stage1_ae.log"
STAGE1_CMD = [
    sys.executable, str(AFTER_DIR/"train_autoencoder.py"),
    "--config", "<GIN_CONFIG>",           # e.g. configs/rave_slakh.gin — pick from §3 output
    "--data", str(CACHE_DIR/"paired.lmdb"),
    "--ckpt-dir", str(CKPT_DIR/"ae"),
    "--sample-rate", str(CFG["sample_rate"]),
    "--batch-size", str(CFG["ae_batch_size"]),
    "--steps", str(CFG["ae_steps"]),
    "--lr", str(CFG["ae_lr"]),
    "--num-workers", str(CFG["num_workers"]),
    # "--resume", "<CKPT_DIR>/ae/latest.pt",  # add this on restart
]
print("STAGE 1 command (edit before launching):", " ".join(STAGE1_CMD))

# Uncomment once the command is correct:
# with open(STAGE1_LOG, "wb") as f:
#     subprocess.Popen(STAGE1_CMD, stdout=f, stderr=subprocess.STDOUT, cwd=str(AFTER_DIR),
#                      stdin=subprocess.DEVNULL, start_new_session=True)
# print("stage 1 launched — tail with: !tail -f", STAGE1_LOG)


In [ ]:
# STAGE 2 — Latent diffusion. Runs on top of the frozen Stage 1 autoencoder.
STAGE2_LOG = ROOT/"stage2_diff.log"
STAGE2_CMD = [
    sys.executable, str(AFTER_DIR/"train_diffusion.py"),
    "--config", "<GIN_CONFIG>",                 # e.g. configs/diffusion_slakh.gin
    "--ae-ckpt", str(CKPT_DIR/"ae"/"best.pt"),   # frozen encoder from Stage 1
    "--data", str(CACHE_DIR/"paired.lmdb"),
    "--ckpt-dir", str(CKPT_DIR/"diff"),
    "--batch-size", str(CFG["diff_batch_size"]),
    "--steps", str(CFG["diff_steps"]),
    "--lr", str(CFG["diff_lr"]),
    "--num-workers", str(CFG["num_workers"]),
    # "--resume", "<CKPT_DIR>/diff/latest.pt",  # add on restart
]
print("STAGE 2 command (edit before launching):", " ".join(STAGE2_CMD))


## 5 · (Optional) Sim-to-real fine-tune on URMP + GuitarSet

The pure-synth model above will have an audible "MIDI-y" or "synthy" quality on real acoustic input. Multiple papers (Zehren et al. `arXiv:2407.19823`, `arXiv:2503.07352`) show that a short fine-tune on real recordings closes most of this gap without hurting the pretrain.

**Skip if evaluation shows the pure-synth model already sounds acceptable.** Otherwise, ~100k more steps on URMP + GuitarSet + a mild synth-batch replay (to avoid catastrophic forgetting) recovers the "air" that Kontakt renders don't capture.

In [ ]:
# URMP is already available on the pod if you ran train_ddsp_48k.ipynb earlier
# (at /workspace/retone_train/urmp_full/extracted/). Otherwise re-fetch — same Dryad
# routes as that notebook.
URMP_SRC = pathlib.Path("/workspace/retone_train/urmp_full/extracted")
if URMP_SRC.exists():
    print("reusing URMP from the DDSP notebook at", URMP_SRC)
else:
    print("URMP not found — copy the fetch cell from train_ddsp_48k.ipynb §2.0 to grab it.")

# Fine-tune command — same shape as Stage 2, but points at URMP-derived LMDB + starts
# from the best diffusion checkpoint.
FT_LOG = ROOT/"finetune_urmp.log"
FT_CMD = [
    sys.executable, str(AFTER_DIR/"train_diffusion.py"),
    "--config", "<GIN_CONFIG_FINETUNE>",
    "--ae-ckpt", str(CKPT_DIR/"ae"/"best.pt"),
    "--data", str(CACHE_DIR/"urmp_paired.lmdb"),
    "--ckpt-dir", str(CKPT_DIR/"diff_finetune"),
    "--resume", str(CKPT_DIR/"diff"/"best.pt"),
    "--batch-size", str(CFG["diff_batch_size"]),
    "--steps", str(CFG["finetune_steps"]),
    "--lr", "3e-5",                                # lower LR to avoid overwriting Stage 2 learning
]
print("fine-tune command (edit before launching):", " ".join(FT_CMD))


## 6 · Export → drop into the worker

AFTER exports to TorchScript via `export_stream.ts` (real-time) or `export.ts` (offline). We want the **non-streaming offline export** because the RunPod worker is batch-render, not real-time — offline export keeps the reverb tail and simplifies inference.

In [ ]:
# Export: produce ae.ts and diff.ts TorchScript artifacts.
FINAL_AE   = OUT_DIR/f"{FAMILY_NAME}_ae.ts"
FINAL_DIFF = OUT_DIR/f"{FAMILY_NAME}_diff.ts"

# The export script is at AFTER_DIR/export.py (or export_stream.py for real-time).
# Confirm the file and its CLI:
export_scripts = sorted(p.name for p in AFTER_DIR.glob("export*.py"))
print("export scripts in AFTER:", export_scripts)

# Template — fill in once AFTER's export CLI is confirmed:
# python export.py --ae-ckpt <best_ae> --diff-ckpt <best_diff_or_finetune> \
#                  --out-ae <FINAL_AE> --out-diff <FINAL_DIFF>
print("\nartifact targets:")
print("  autoencoder :", FINAL_AE)
print("  diffusion   :", FINAL_DIFF)

# Sanity load: verify both files torch.jit.load without errors on this pod, before scp'ing back.
# import torch
# ae   = torch.jit.load(str(FINAL_AE)).eval()
# diff = torch.jit.load(str(FINAL_DIFF)).eval()
# print("both models loaded OK; ae params:",   sum(p.numel() for p in ae.parameters()))
# print("                     ; diff params:", sum(p.numel() for p in diff.parameters()))


## 7 · Drop into the worker (run on your laptop, not the pod)

`scp` both `.ts` files back, then in your local retone repo:

```bash
mkdir -p runpod/after_models/piano_orchestral
scp -i ~/.ssh/retone_runpod_ed25519 -P 11257 \
  root@<pod-ip>:/workspace/retone_train_after/artifacts/piano_orchestral/piano_orchestral_ae.ts   runpod/after_models/piano_orchestral/ae.ts
scp -i ~/.ssh/retone_runpod_ed25519 -P 11257 \
  root@<pod-ip>:/workspace/retone_train_after/artifacts/piano_orchestral/piano_orchestral_diff.ts runpod/after_models/piano_orchestral/diff.ts
```

Whitelist the new weights past `.gitignore`:

```
!runpod/after_models/piano_orchestral/ae.ts
!runpod/after_models/piano_orchestral/diff.ts
```

**Register the family** in the worker + backend. Both changes are one-line-per-list additions:

- `runpod/after_engine.py` (created by Track A): add `"piano_orchestral": ("piano_orchestral/ae.ts", "piano_orchestral/diff.ts")` to `MODELS`.
- `backend/app/services/tone_transfer.py`: add `"piano_orchestral"` to `AFTER_INSTRUMENTS`.

Then commit + push (the diffusion `.ts` is likely 100-500 MB — the same large-file caveat from `train_ddsp_48k.ipynb` applies: `git config http.version HTTP/1.1` and use `git -c protocol.version=0 push` if GitHub 408s. If any single file exceeds the 100 MB hard cap, use Git LFS or fetch at Dockerfile build time — see `runpod/download_models.py` for the pattern.)

RunPod's GitHub integration rebuilds the worker image on push. Once green, the frontend picks up the new option (`after:piano_orchestral`) automatically via `/meta/after-instruments`.

## 8 · Verification (once the checkpoint is deployed)

1. **Local sanity** — before shipping, run the exported `.ts` on this pod against a held-out Slakh piano stem: confirm the output is real audio (RMS > 0.02, peak > 0.2) and audibly orchestral.
2. **Pod smoke test** — trigger `tone_transfer` from the backend with `engine: "after"`, `instrument: "piano_orchestral"` on an existing piano stem (e.g. `stems/ac344df53cf9/piano.flac`). Confirm the presigned output URL returns real orchestral-timbred audio of the same duration as the input.
3. **A/B against pretrained AFTER** — same input piano stem, our checkpoint vs the pretrained Slakh-a2a checkpoint from Track A. Our own-trained should be at least on par (same architecture, similar data) and win on orchestral targets that pretrained AFTER doesn't cover.
4. **Frontend E2E** — pick a piano stem in the DAW, select "AFTER · piano → orchestral" from the dropdown, hear the swap.
5. **Sim-to-real gap check (if §5 was run)** — same test on real (URMP) piano input, not Slakh's Kontakt renders. Un-fine-tuned checkpoint should sound noticeably worse than fine-tuned on real audio; if not, §5 didn't help and can be skipped for future family trainings.

## 9 · Known risks + follow-ups

- **AFTER's gin configs are the actual load-bearing surface.** The CLI templates in this notebook are placeholders — every command references a config file (`--config configs/*.gin`) whose exact keys must be picked once the repo is cloned. Plan on 30-60 min in §3 to inspect and fill in.
- **Compatibility patches** — AFTER pins torch 2.0+ so it should be modern-safe, unlike the DDSP codebase. If a training step crashes on omegaconf, pandas, or torch.fft, cross-reference the DDSP notebook's PATCH cell — the same fixes may apply.
- **`--num-workers` right-sizing** — the DDSP notebook found the pod's cgroup CPU quota (7.65 cores) was much lower than `nproc` reported (48). If AFTER's dataloader saturates the CPU before the GPU, drop `num_workers` and confirm sustained samples/sec via a 90-second window (not a short post-restart snapshot).
- **Diffusion `.ts` file size** — likely 100-500 MB. Pre-verify with `wc -c` before `git push`; if > 100 MB, add to `runpod/download_models.py` for build-time fetch instead of committing.
- **Follow-up family: guitar → orchestral** — same notebook, change `SOURCE_STEMS = ["Guitar"]` in §0. AFTER can share the autoencoder across families (retrain diffusion only) — worth a follow-up ablation once piano is validated.
- **License note** — the AFTER *code* is CC BY-NC 4.0, but our own-trained weights (this notebook's output) are 100% ours. Track A's pretrained weights are the CC BY-NC dependency; this notebook's Track B output removes that dependency for our shipping path.
